# 10-phase11-scale-corrected-init

**neuron Phase 11** — Phase 10 의 결정적 발견 ("sweet spot magnitude 는 자유도의 의미적 위치에
의존") 의 빠른 검증. ChannelGraphLinear 의 adj 에 **scale-corrected init** 적용.

핵심 가설:
1. **uniform_around_one ≈ channel_full** — 1.0 근처 noise 도 magnitude 균형 유지 + adj 학습 활성
2. **uniform_around_one > uniform_small** — magnitude rule 직접 입증 (small=10% scale vs around_one=100% scale)
3. **plain ↔ channel_full ↔ uniform_around_one 모두 ≈** — graph 구조의 free 성 + scale-corrected init 동등성

설계: 4-way × 2 seed = 8 run, max_steps=1500.
- arch ∈ {plain, channel_full, channel_uniform_small, channel_uniform_around_one}
- seed ∈ {42, 123}

데이터: TinyShakespeare (char-LM)
시드: [42, 123]
작성일: 2026-05-26
연관: Issue [#63](https://github.com/EinSofINTEREST/GraphLM/issues/63) / Phase 10 baseline PR [#62](https://github.com/EinSofINTEREST/GraphLM/pull/62)

## 1. 환경

In [ ]:
import logging
import sys

import torch

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.graph_channel_demo import train_channel_graph_mlp
from graphlm.utils import repo_root

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)
print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)

## 2. 실험 설정

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase11"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123]
ARCHS = ["plain", "channel_full", "channel_uniform_small", "channel_uniform_around_one"]
EMB_DIM = 64
HIDDEN_DIM = 256
N_GRAM = 4
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 1500

# Phase 10 baseline (PR #62)
PHASE10_PLAIN_MEAN = 2.1487
PHASE10_CHANNEL_FULL_MEAN = 2.1339
PHASE10_CHANNEL_UNIFORM_SMALL_MEAN = 2.3268  # +0.18 열위 — magnitude scale 함정

print(f"SEEDS      = {SEEDS}")
print(f"ARCHS      = {ARCHS}")
print(f"HIDDEN_DIM = {HIDDEN_DIM}")
print(f"MAX_STEPS  = {MAX_STEPS}")
print(f"전체 run   = {len(SEEDS) * len(ARCHS)}")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
V = tokenizer.vocab_size
print(f"vocab_size : {V}")

## 4. sweep 학습

In [ ]:
runs = {}
for seed in SEEDS:
    for arch in ARCHS:
        key = (seed, arch)
        print(f"--- seed={seed}, arch={arch} ---")
        runs[key] = train_channel_graph_mlp(
            dataset=dataset,
            vocab_size=V,
            seed=seed,
            arch=arch,
            emb_dim=EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            n_gram=N_GRAM,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_steps=MAX_STEPS,
            device=DEVICE,
        )
        print(f"  done: final_loss={runs[key]['final_loss']:.4f}")
        if str(DEVICE).startswith("cuda"):
            torch.cuda.empty_cache()

## 5. 결과 표 + Phase 10 baseline 비교

In [ ]:
import statistics

print(f"{'arch':>32}  {'seed':>5}  {'final_loss':>11}")
print("-" * 60)
for arch in ARCHS:
    for seed in SEEDS:
        r = runs[(seed, arch)]
        print(f"{arch:>32}  {seed:>5}  {r['final_loss']:>11.4f}")

print()
print("=== arch 별 mean ===")
agg = {}
for arch in ARCHS:
    fls = [runs[(s, arch)]["final_loss"] for s in SEEDS]
    agg[arch] = dict(mean=statistics.mean(fls), range=max(fls) - min(fls))
    print(f"  {arch:>32}: mean={agg[arch]['mean']:.4f}, range={agg[arch]['range']:.4f}")

print()
print("=== Phase 10 baseline (PR #62) 비교 ===")
print(f"  Phase 10 plain                        : {PHASE10_PLAIN_MEAN:.4f}")
print(f"  Phase 10 channel_full                 : {PHASE10_CHANNEL_FULL_MEAN:.4f}")
print(
    f"  Phase 10 channel_uniform_small        : {PHASE10_CHANNEL_UNIFORM_SMALL_MEAN:.4f}  (+0.18 열위 — magnitude 함정)"
)
print()
print(f"  Phase 11 plain (재현)                  : {agg['plain']['mean']:.4f}")
print(f"  Phase 11 channel_full (재현)           : {agg['channel_full']['mean']:.4f}")
print(f"  Phase 11 channel_uniform_small (재현)  : {agg['channel_uniform_small']['mean']:.4f}")
print(
    f"  Phase 11 channel_uniform_around_one   : {agg['channel_uniform_around_one']['mean']:.4f}  ← 핵심 검증"
)

print()
print("=== 자동 verdict ===")
plain = agg["plain"]["mean"]
full = agg["channel_full"]["mean"]
around_one = agg["channel_uniform_around_one"]["mean"]
small = agg["channel_uniform_small"]["mean"]
range_max = max(agg[a]["range"] for a in ARCHS)
print(f"  range_max = {range_max:.4f}")
print(f"  around_one vs plain          : {around_one - plain:+.4f}")
print(f"  around_one vs channel_full   : {around_one - full:+.4f}")
print(f"  around_one vs uniform_small  : {around_one - small:+.4f}  (음수면 magnitude rule 입증)")
if around_one - small < -0.05:
    print("  ✅ magnitude rule 입증 — around_one 이 uniform_small 보다 명확히 우위")
if abs(around_one - full) < range_max:
    print("  ✅ around_one ≈ channel_full — scale balance 작동")

## 6. 학습된 adj distribution 비교 — 4 arch

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# channel_* 3 arch 의 fc1 adj 분포 비교
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
arch_names = ["channel_full", "channel_uniform_small", "channel_uniform_around_one"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for i, arch in enumerate(arch_names):
    r = runs[(SEEDS[0], arch)]
    if r["final_adj"] is None:
        continue
    adj = r["final_adj"]["fc1"].numpy().flatten()
    axes[i].hist(adj, bins=80, alpha=0.7, color=colors[i])
    axes[i].set_xlabel("adj value")
    axes[i].set_ylabel("count")
    axes[i].set_title(f"{arch}\nmean={adj.mean():.3f}, std={adj.std():.3f}")
    axes[i].axvline(0, color="red", linestyle="--", lw=0.8, alpha=0.5)
    axes[i].axvline(adj.mean(), color="green", linestyle=":", lw=1)
    axes[i].grid(alpha=0.3)

fig.suptitle(f"Phase 11 — channel adj 분포 비교 (seed={SEEDS[0]}, fc1)", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "adj_dist_compare.png", dpi=120)
plt.show()

## 7. loss curve 비교 — 4 arch mean ± σ

In [ ]:
window = 30
colors = {
    "plain": "#1f77b4",
    "channel_full": "#2ca02c",
    "channel_uniform_small": "#ff7f0e",
    "channel_uniform_around_one": "#d62728",
}

fig, ax = plt.subplots(figsize=(13, 5))
for arch in ARCHS:
    seed_curves = []
    for seed in SEEDS:
        losses = runs[(seed, arch)]["losses"]
        smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
        seed_curves.append(smoothed)
    arr = np.array(seed_curves)
    steps = np.arange(window - 1, window - 1 + arr.shape[1])
    mean = arr.mean(axis=0)
    std = arr.std(axis=0, ddof=1)
    ax.plot(steps, mean, color=colors[arch], lw=1.5, label=arch)
    ax.fill_between(steps, mean - std, mean + std, color=colors[arch], alpha=0.15)
ax.axhline(
    PHASE10_CHANNEL_FULL_MEAN,
    color="gray",
    linestyle=":",
    lw=1,
    alpha=0.7,
    label=f"Phase 10 channel_full ({PHASE10_CHANNEL_FULL_MEAN})",
)
ax.set_xlabel("step")
ax.set_ylabel(f"loss (smoothed window={window})")
ax.set_title("Phase 11 — 4 arch loss curve (mean ± σ across 2 seeds)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "loss_curves.png", dpi=120)
plt.show()

## 결과 요약 / Phase 12 권장

확인 포인트:
- §5 around_one vs uniform_small (마이너스 시 magnitude rule 입증)
- §5 around_one vs channel_full (range 보다 작은 차이 시 scale balance 작동)
- §6 around_one 의 adj 분포 — 1.0 근처 spread 가 학습 후 어떻게 변화?
- §7 loss curve — around_one 이 uniform_small 보다 명확히 낮은 위치?

**판정 시나리오**:
- **A. around_one ≈ channel_full + uniform_small 명확 열위 유지** ⭐ — magnitude rule 확정, Phase 12 (hybrid) 진입
- **B. around_one > channel_full** — noise 가 학습 가속 (implicit pruning 활성), surprise positive
- **C. around_one ≈ uniform_small** — magnitude 만으로 부족, 다른 요인 (예: weight 와의 동시 학습 dynamics)

**참고**:
- Phase 10 결과: https://www.notion.so/36ce8b70b7aa81ff82a6edd3e2d03770
- 아키텍처 구성 계획: https://www.notion.so/36ce8b70b7aa818cbf1fe71687b449b8